# MahimandalSR benchmark

Inspect saved results first. Fetching, training and test evaluation are separate opt-in steps. All runs use full160 inputs; figures use display-only stretches.


In [ ]:
from pathlib import Path
import sys, subprocess

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "benchmark/run.py").is_file():
    raise RuntimeError("Open this notebook from the mahimandal-review folder")
DATASET = ROOT / "data/reflectance16"
SOURCES = ROOT / "models/upstream"
OUTPUT = ROOT / "runs"
DEVICE = "cpu"  # use cuda only on your assigned GPU

def run(*arguments):
    subprocess.run([sys.executable, str(ROOT / "benchmark/run.py"),
                    *map(str, arguments)], cwd=str(ROOT), check=True)


In [ ]:
import pandas as pd
for modality in ("rgb", "rgbnir"):
    for path in sorted((ROOT / "results/benchmark/reflectance16" / modality).glob("*/seed_*/evaluation/test/*/metrics.json")):
        import json
        scores = json.loads(path.read_text())
        print(scores["model"], modality, "PSNR", round(scores["psnr"], 4), "SSIM", round(scores["ssim"], 4))


In [ ]:
MODELS = ["EDSR", "RCAN", "RDN", "CAFRN", "DCM", "FENet", "RealESRGAN_SRVGG_X3"]
MODALITIES = ["rgb", "rgbnir"]
FETCH_SOURCES = False
RUN_TRAINING = False
RUN_VALIDATION = False
RUN_TEST = False  # enable only after validation selection is finished
if FETCH_SOURCES:
    run("fetch", "--sources", SOURCES)


In [ ]:
import json
registry = json.loads((ROOT / "benchmark/registry.json").read_text())
for modality in MODALITIES:
    for model in MODELS:
        if modality not in registry[model]["modalities"]:
            continue
        if RUN_TRAINING:
            run("smoke", "--model", model, "--modality", modality, "--sources", SOURCES,
                "--output", OUTPUT, "--device", DEVICE, "--backward")
            run("train", "--model", model, "--dataset", DATASET, "--sources", SOURCES,
                "--output", OUTPUT, "--radiometry", "reflectance16", "--modality", modality,
                "--epochs", 50, "--patience", 3, "--batch-size", 1, "--train-lr-crop", 160,
                "--lr", 0.0001, "--seed", 42, "--device", DEVICE)
        for split, enabled in (("val", RUN_VALIDATION), ("test", RUN_TEST)):
            if enabled:
                run("evaluate", "--model", model, "--dataset", DATASET, "--sources", SOURCES,
                    "--output", OUTPUT, "--radiometry", "reflectance16", "--modality", modality,
                    "--train-lr-crop", 160, "--seed", 42, "--split", split,
                    "--border-crop", 3, "--device", DEVICE)


In [ ]:
# Bicubic has no training stage.
if RUN_TEST:
    for modality in MODALITIES:
        run("evaluate", "--model", "bicubic", "--dataset", DATASET, "--sources", SOURCES,
            "--output", OUTPUT, "--radiometry", "reflectance16", "--modality", modality,
            "--train-lr-crop", 160, "--seed", 42, "--split", "test", "--border-crop", 3,
            "--device", DEVICE)
